# Tutorial 1: Basic Multi-Objective Optimization with nsgablack

This notebook introduces the basics of multi-objective optimization using nsgablack. We'll cover:
- Understanding multi-objective optimization
- Setting up optimization problems
- Using NSGA-II algorithm
- Analyzing results
- Visualization

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add nsgablack to path
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from nsgablack import ZDT1BlackBox, BlackBoxSolverNSGAII

# Set up matplotlib for better plots
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

## What is Multi-Objective Optimization?

Multi-objective optimization involves optimizing multiple conflicting objectives simultaneously. Unlike single-objective optimization where we look for a single best solution, in multi-objective optimization we seek a set of solutions representing different trade-offs between objectives.

The set of optimal solutions is called the **Pareto front** - solutions where you cannot improve one objective without worsening another.

In [ ]:
# Let's visualize this concept with a simple example
def visualize_concept():
    """Visualize the concept of Pareto optimality."""
    # Generate sample solutions
    np.random.seed(42)
    x = np.random.uniform(0, 1, (100, 2))
    
    # Calculate objectives (conflicting)
    f1 = x[:, 0]  # Minimize x0
    f2 = 1 - np.sqrt(x[:, 0]) + 0.1 * (x[:, 1] - 0.5)**2  # Related to x1
    
    # Find Pareto front (simplified)
    pareto_indices = []
    for i in range(len(x)):
        is_pareto = True
        for j in range(len(x)):
            if i != j:
                if (f1[j] <= f1[i] and f2[j] <= f2[i]) and (f1[j] < f1[i] or f2[j] < f2[i]):
                    is_pareto = False
                    break
        if is_pareto:
            pareto_indices.append(i)
    
    plt.figure(figsize=(12, 5))
    
    # Plot all solutions
    plt.subplot(1, 2, 1)
    plt.scatter(f1, f2, c='lightblue', alpha=0.6, s=50, label='All solutions')
    plt.scatter(f1[pareto_indices], f2[pareto_indices], c='red', s=100, 
               label='Pareto optimal solutions', edgecolors='darkred', linewidths=2)
    plt.xlabel('Objective 1 (minimize)')
    plt.ylabel('Objective 2 (minimize)')
    plt.title('Pareto Front Concept')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    # Show trade-offs
    plt.subplot(1, 2, 2)
    pareto_solutions = x[pareto_indices]
    plt.scatter(pareto_solutions[:, 0], pareto_solutions[:, 1], 
               c=np.arange(len(pareto_solutions)), cmap='viridis', s=100)
    plt.xlabel('Variable 0')
    plt.ylabel('Variable 1')
    plt.title('Pareto Solutions in Decision Space')
    plt.colorbar(label='Solution index')
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Total solutions: {len(x)}")
    print(f"Pareto optimal solutions: {len(pareto_indices)}")
    print(f"Pareto ratio: {len(pareto_indices)/len(x):.2%}")

visualize_concept()

## Standard Test Problems

nsgablack includes several standard test problems:
- **ZDT1**: Continuous convex Pareto front
- **ZDT3**: Disconnected Pareto front
- **DTLZ2**: Scalable to many objectives
- **Sphere**: Simple single/multi-objective test

Let's start with ZDT1, the most common test problem.

In [ ]:
# Create and examine a ZDT1 problem
problem = ZDT1BlackBox(dimension=10)

print("Problem Information:")
print(f"  Name: {problem.name}")
print(f"  Dimension: {problem.dimension}")
print(f"  Bounds: {problem.bounds}")

# Test the evaluation function
test_solution = np.random.random(10)
objectives = problem.evaluate(test_solution)
print(f"\nTest solution: {test_solution}")
print(f"Objectives: {objectives}")
print(f"Number of objectives: {len(objectives)}")

## NSGA-II Algorithm

NSGA-II (Non-dominated Sorting Genetic Algorithm II) is one of the most popular multi-objective optimization algorithms. It uses:
- Fast non-dominated sorting to classify solutions
- Crowding distance to maintain diversity
- Genetic operators (crossover and mutation)

In [ ]:
# Set up and run NSGA-II
solver = BlackBoxSolverNSGAII(problem)

# Configure algorithm parameters
solver.pop_size = 100          # Population size
solver.max_generations = 200   # Number of generations
solver.crossover_rate = 0.9     # Crossover probability
solver.mutation_rate = 0.1      # Mutation probability
solver.random_seed = 42         # For reproducible results

print("NSGA-II Configuration:")
print(f"  Population size: {solver.pop_size}")
print(f"  Max generations: {solver.max_generations}")
print(f"  Crossover rate: {solver.crossover_rate}")
print(f"  Mutation rate: {solver.mutation_rate}")

# Run the optimization
print("\nRunning NSGA-II optimization...")
result = solver.run()

print("\nOptimization completed!")
print(f"Generations: {result['generations']}")
print(f"Total evaluations: {result['evaluations']}")
print(f"Pareto solutions found: {len(result['pareto_solutions'])}")
print(f"Converged: {result.get('converged', 'Unknown')}")
print(f"Elapsed time: {result['elapsed_time']:.2f} seconds")

In [ ]:
# Analyze the results
pareto_solutions = result['pareto_solutions']
pareto_objectives = result['pareto_objectives']

print("\nResult Analysis:")
print(f"Pareto front size: {len(pareto_solutions)}")

# Objective statistics
if len(pareto_objectives) > 0:
    obj1 = pareto_objectives[:, 0]
    obj2 = pareto_objectives[:, 1]
    print(f"\nObjective 1 range: [{np.min(obj1):.4f}, {np.max(obj1):.4f}]")
    print(f"Objective 2 range: [{np.min(obj2):.4f}, {np.max(obj2):.4f}]")
    print(f"Objective 1 mean: {np.mean(obj1):.4f}")
    print(f"Objective 2 mean: {np.mean(obj2):.4f}")
    
    # Diversity metrics
    diversity = np.mean([np.linalg.norm(pareto_objectives[i] - pareto_objectives[j]) 
                           for i in range(len(pareto_objectives))
                           for j in range(i+1, len(pareto_objectives))])
    print(f"\nDiversity (average distance): {diversity:.4f}")

In [ ]:
# Visualize the results
def plot_pareto_front(objectives, title="Pareto Front"):
    """Plot the Pareto front."""
    plt.figure(figsize=(10, 6))
    
    # Plot Pareto front
    plt.scatter(objectives[:, 0], objectives[:, 1], 
               alpha=0.7, s=50, c='blue', edgecolors='darkblue')
    
    plt.xlabel('Objective 1')
    plt.ylabel('Objective 2')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    
    # Add reference point
    ref_point = np.max(objectives, axis=0) * 1.1
    plt.scatter([ref_point[0]], [ref_point[1]], c='red', s=100, 
               marker='*', label='Reference point')
    plt.legend()
    
    plt.show()

def plot_convergence_history(history):
    """Plot convergence history if available."""
    if history and 'convergence' in history:
        plt.figure(figsize=(12, 4))
        
        plt.subplot(1, 2, 1)
        plt.plot(history['convergence'])
        plt.xlabel('Generation')
        plt.ylabel('Convergence Metric')
        plt.title('Convergence History')
        plt.grid(True, alpha=0.3)
        
        if 'diversity' in history and len(history['diversity']) > 0:
            plt.subplot(1, 2, 2)
            plt.plot(history['diversity'])
            plt.xlabel('Generation')
            plt.ylabel('Diversity Metric')
            plt.title('Diversity History')
            plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    else:
        print("No convergence history available")

# Plot results
plot_pareto_front(pareto_objectives, "ZDT1 Pareto Front")
plot_convergence_history(result.get('history', {}))

## Solution Selection

From the Pareto front, you might want to select a specific solution based on your preferences:

In [ ]:
# Solution selection strategies
def select_solutions(pareto_solutions, pareto_objectives, strategy="knee"):
    """Select specific solutions from Pareto front."""
    if strategy == "extreme":
        # Select extreme solutions
        obj1_min_idx = np.argmin(pareto_objectives[:, 0])
        obj2_min_idx = np.argmin(pareto_objectives[:, 1])
        return {
            "best_obj1": pareto_solutions[obj1_min_idx],
            "best_obj2": pareto_solutions[obj2_min_idx],
            "best_obj1_obj": pareto_objectives[obj1_min_idx],
            "best_obj2_obj": pareto_objectives[obj2_min_idx]
        }
    
    elif strategy == "compromise":
        # Select compromise solution (closest to ideal point)
        ideal_point = np.min(pareto_objectives, axis=0)
        distances = [np.linalg.norm(obj - ideal_point) for obj in pareto_objectives]
        compromise_idx = np.argmin(distances)
        return {
            "compromise": pareto_solutions[compromise_idx],
            "compromise_obj": pareto_objectives[compromise_idx],
            "distance_to_ideal": distances[compromise_idx]
        }
    
    elif strategy == "knee":
        # Simple knee point detection
        # Normalize objectives
        norm_obj = (pareto_objectives - np.min(pareto_objectives, axis=0)) / (
                    np.max(pareto_objectives, axis=0) - np.min(pareto_objectives, axis=0))
        
        # Sort by first objective
        sorted_idx = np.argsort(norm_obj[:, 0])
        sorted_obj = norm_obj[sorted_idx]
        
        # Find knee point (simplified)
        max_knee_angle = -1
        knee_idx = 0
        
        for i in range(1, len(sorted_obj) - 1):
            # Calculate angle at point i
            v1 = sorted_obj[i] - sorted_obj[i-1]
            v2 = sorted_obj[i+1] - sorted_obj[i]
            if np.linalg.norm(v1) > 0 and np.linalg.norm(v2) > 0:
                cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
                angle = np.arccos(np.clip(cos_angle, -1, 1))
                if angle > max_knee_angle:
                    max_knee_angle = angle
                    knee_idx = i
        
        original_idx = sorted_idx[knee_idx]
        return {
            "knee": pareto_solutions[original_idx],
            "knee_obj": pareto_objectives[original_idx],
            "knee_angle": np.degrees(max_knee_angle)
        }
    
    else:
        return {}

# Select different types of solutions
selected = select_solutions(pareto_solutions, pareto_objectives, "compromise")
extreme = select_solutions(pareto_solutions, pareto_objectives, "extreme")
knee = select_solutions(pareto_solutions, pareto_objectives, "knee")

print("\nSolution Selection:")
print(f"Compromise solution: {selected['compromise_obj']}")
print(f"Extreme solutions: best_obj1={extreme['best_obj1_obj']}, best_obj2={extreme['best_obj2_obj']}")
if 'knee_angle' in knee:
    print(f"Knee solution: {knee['knee_obj']} (angle: {knee['knee_angle']:.1f}°)")

## Interactive Solution Explorer

Let's create an interactive tool to explore the Pareto front:

In [ ]:
# Interactive solution explorer (if in Jupyter)
try:
    from ipywidgets import interact, FloatSlider, SelectionSlider
    import plotly.graph_objects as go
    import plotly.express as px
    
    def create_interactive_plot():
        def plot_with_highlighting(solution_idx):
            fig = go.Figure()
            
            # Plot all solutions
            fig.add_trace(go.Scatter(
                x=pareto_objectives[:, 0],
                y=pareto_objectives[:, 1],
                mode='markers',
                marker=dict(
                    size=8,
                    color='lightblue',
                    line=dict(width=1, color='blue')
                ),
                name='Pareto Solutions',
                opacity=0.7
            ))
            
            # Highlight selected solution
            if 0 <= solution_idx < len(pareto_objectives):
                fig.add_trace(go.Scatter(
                    x=[pareto_objectives[solution_idx, 0]],
                    y=[pareto_objectives[solution_idx, 1]],
                    mode='markers',
                    marker=dict(
                        size=15,
                        color='red',
                        line=dict(width=3, color='darkred')
                    ),
                    name='Selected Solution'
                ))
            
            fig.update_layout(
                title='Interactive Pareto Front Explorer',
                xaxis_title='Objective 1',
                yaxis_title='Objective 2',
                showlegend=True
            )
            
            # Show solution details
            if 0 <= solution_idx < len(pareto_solutions):
                solution = pareto_solutions[solution_idx]
      
                # Create a detailed text annotation
                details = f"""<b>Solution {solution_idx}</b><br>"""
                for i, (val, (min_val, max_val)) in enumerate(zip(solution, list(problem.bounds.values())):
                    details += f"<br><b>Var {i}:</b> {val:.4f} [{min_val}, {max_val}]"
                details += f"""<br><br><b>Objectives:</b><br>"""
                details += f"<br><b>Obj 1:</b> {pareto_objectives[solution_idx, 0]:.4f}"
                details += f"<br><b>Obj 2:</b> {pareto_objectives[solution_idx, 1]:.4f}""""
                
                fig.add_annotation(
                    x=pareto_objectives[solution_idx, 0],
                    y=pareto_objectives[solution_idx, 1],
                    text=details,
                    showarrow=False,
                    align='left',
                    bgcolor="white",
                    bordercolor="black",
                    borderwidth=1
                )
            
            fig.show()
        
        interact(plot_with_highlighting, solution_idx=(0, len(pareto_objectives)-1, 1))
        
    create_interactive_plot()
    
except ImportError:
        print("Interactive widgets not available (not in Jupyter or ipywidgets not installed)")
        print("Falling back to static plot...")
        plot_pareto_front(pareto_objectives, "Pareto Front - Select any solution to explore")

## Performance Metrics

Let's calculate some performance metrics for our optimization:

In [ ]:
def calculate_hypervolume(objectives, reference_point=None):
    """Calculate hypervolume indicator (2D case)."""
    if len(objectives) == 0:
        return 0.0
    
    if reference_point is None:
        reference_point = np.max(objectives, axis=0) * 1.1
    
    # Sort by first objective
    sorted_idx = np.argsort(objectives[:, 0])
    sorted_obj = objectives[sorted_idx]
    
    # Calculate hypervolume
    volume = 0.0
    for i in range(len(sorted_obj)):
        if i == 0:
            volume += (reference_point[0] - sorted_obj[i, 0]) * (reference_point[1] - sorted_obj[i, 1])
        else:
            volume += (reference_point[0] - sorted_obj[i, 0]) * (sorted_obj[i-1, 1] - sorted_obj[i, 1])
    
    return volume

def calculate_spread(objectives):
    """Calculate spread of solutions."""
    if len(objectives) < 2:
        return 0.0
    
    distances = []
    for i in range(len(objectives)):
        for j in range(i+1, len(objectives)):
            distances.append(np.linalg.norm(objectives[i] - objectives[j]))
    
    return np.mean(distances)

def calculate_spacing(objectives):
    """Calculate spacing metric (uniformity)."""
    if len(objectives) < 2:
        return 1.0  # Perfect spacing
    
    # Calculate nearest neighbor distances
    min_distances = []
    for i in range(len(objectives)):
        distances = [np.linalg.norm(objectives[i] - objectives[j]) 
                   for j in range(len(objectives)) if i != j]
        min_distances.append(min(distances))
    
    # Spacing quality (lower standard deviation is better)
    return np.std(min_distances) / np.mean(min_distances) if np.mean(min_distances) > 0 else 0

# Calculate metrics
hypervolume = calculate_hypervolume(pareto_objectives)
spread = calculate_spread(pareto_objectives)
spacing = calculate_spacing(pareto_objectives)

print("\nPerformance Metrics:")
print(f"  Hypervolume: {hypervolume:.4f}")
print(f"  Spread (average distance): {spread:.4f}")
print(f"  Spacing quality (std/mean): {spacing:.4f} (lower is better)")

# Quality assessment
print("\nQuality Assessment:")
if hypervolume > 0.8:  # ZDT1 typically has hypervolume around 1.0
    print(f"  ✓ Hypervolume: Excellent ({hypervolume:.4f})")
elif hypervolume > 0.6:
    print(f"  ✓ Hypervolume: Good ({hypervolume:.4f})")
else:
    print(f"  ✗ Hypervolume: Poor ({hypervolume:.4f})")

if spread > 0.1:
    print(f"  ✓ Spread: Good ({spread:.4f})")
else:
    print(f"  ⚠ Spread: Low ({spread:.4f}) - solutions may be clustered")

if spacing < 0.3:
    print(f"  ✓ Spacing: Excellent ({spacing:.4f})")
elif spacing < 0.5:
    print(f"  ✓ Spacing: Good ({spacing:.4f})")
else:
    print(f"  ⚠ Spacing: Poor ({spacing:.4f}) - solutions may be unevenly distributed")

## What's Next?

Congratulations! You've successfully completed basic multi-objective optimization with nsgablack. Here's what you can explore next:

1. **Try different problems**: Experiment with ZDT3 (disconnected front) or DTLZ2 (3+ objectives)
2. **Use the bias system**: Add domain knowledge and constraints
3. **Parallel evaluation**: Speed up expensive function evaluations
4. **Advanced algorithms**: Try MOEA/D, Bayesian optimization, or VNS
5. **Custom problems**: Define your own optimization problems

Continue to the next tutorial to learn about the bias system, or explore the user guide for more advanced topics!